# HTD1

- Edwin de Leon
- Gustavo Cruz
- Mathew Cordero
- Josue Say



[Hoja de Trabajo Repositorio](https://github.com/donmatthiuz/RL/tree/htd1)

## Task 1

Responda las siguientes preguntas de forma clara, concisa y argumentada. No basta con definir: se espera
que usted conecte el concepto con sus implicaciones prácticas o estratégicas.

### Pregunta 1

Elija un sistema real de su área de interés (puede ser de banca, telecomunicaciones, salud, videojuegos, robótica u otro).

Identifique y justifique para ese sistema:

* (a) qué representa el estado $$S_t$$
* (b) qué acciones componen el espacio $$\mathcal{A}$$
* (c) cómo diseñaría la función de recompensa $$R$$
* (d) si el entorno es totalmente observable o parcialmente observable, y por qué


#### Respuesta 1

**Sistema:** Videojuego (Agente en un Juego de Rol)

Se analiza un sistema basado en un videojuego de rol, donde un agente (IA) debe aprender a combatir. El agente cuenta con el siguiente inventario y habilidades:

| | Ataque | Proteccion | Curacion | Habilidad |
|---|---|---|---|---|
| **Agente (tu)** | Espada, Garrote | (ninguna) | 1 Hierba Roja | Bola de Fuego |
| **Enemigo** | Garrote | Hombreras | 2 Hierbas Rojas | Bola de Fuego |

Reglas del sistema:

- **Espada:** corta, pero no funciona bien contra objetivos que usan hombreras (el dano se reduce casi a cero).
- **Garrote:** golpe contundente que ignora la proteccion (funciona igual con o sin hombreras) y ademas aturde al enemigo un turno, impidiendole atacar.
- **Bola de Fuego:** dano magico que ignora la proteccion fisica (hombreras); ambos personajes la poseen, por lo que es simetrica.
- **Hierba Roja:** cura una cantidad fija de salud; el agente solo tiene 1, mientras que el enemigo tiene 2, lo que representa una desventaja de recursos a largo plazo.
- **Chamarra:** objeto de proteccion adicional disponible en el sistema, aunque en este enfrentamiento concreto el enemigo eligio equiparse con Hombreras en lugar de Chamarra.


#### (a) Estado St

El estado St representa toda la informacion relevante del entorno en el instante t. Para este sistema concreto, el estado debe capturar no solo la salud de ambos personajes, sino tambien que arma y proteccion tiene equipada cada uno, y el estado de sus recursos:

$$
S_t = \{ salud\_propia_t,\ salud\_enemigo_t,\ arma\_equipada\_propia_t,\ proteccion\_equipada\_enemigo_t,\ hierbas\_rojas\_propias_t,\ hierbas\_rojas\_enemigo_t,\ cooldown\_bola\_fuego\_propia_t,\ cooldown\_bola\_fuego\_enemigo_t,\ aturdido_t \}
$$

Donde, por ejemplo:

- `proteccion_equipada_enemigo_t = hombreras` indica que atacar con Espada sera poco efectivo ese turno.
- `hierbas_rojas_propias_t = 1` y `hierbas_rojas_enemigo_t = 2` reflejan la asimetria de recursos de curacion.
- `aturdido_t` indica si alguno de los dos personajes no puede actuar en ese turno (efecto del Garrote).

**Justificacion:**
El agente necesita conocer no solo "cuanta salud tiene el enemigo", sino especificamente que proteccion lleva puesta, porque esa informacion determina si conviene usar Espada, Garrote o Bola de Fuego. De la misma forma, conocer cuantas Hierbas Rojas le quedan a cada uno permite planificar si conviene alargar o acortar el combate.


#### (b) Espacio de Acciones A

El espacio de acciones incluye las acciones concretas disponibles segun el inventario del agente:

$$
\mathcal{A} = \{ atacar\_espada,\ atacar\_garrote,\ usar\_bola\_de\_fuego,\ usar\_hierba\_roja,\ huir,\ no\_hacer\_nada \}
$$

De forma estructurada, cada accion se puede describir con parametros:

$$
\mathcal{A} = \{ tipo\_accion,\ objetivo \}
$$

donde:

- `tipo_accion` in { ataque_espada, ataque_garrote, habilidad_bola_fuego, curacion_hierba, huir }
- `objetivo` in { enemigo, propio }

Cada `tipo_accion` tiene efectos distintos definidos por las reglas del sistema:

- `ataque_espada`: dano alto si el objetivo no tiene hombreras; dano casi nulo si el objetivo si las tiene.
- `ataque_garrote`: dano moderado, ignora proteccion, y aturde al objetivo un turno.
- `habilidad_bola_fuego`: dano moderado-alto, ignora proteccion, sin aturdimiento.
- `curacion_hierba`: recupera salud, mas valiosa cuanto menor sea la cantidad restante (recurso escaso para el agente, que solo tiene 1).

**Justificacion:**
Definir las acciones a este nivel de detalle es necesario porque, a diferencia de un espacio de acciones generico ("atacar" o "defender"), aqui la efectividad de cada accion depende directamente del estado del oponente (si tiene o no hombreras equipadas). El agente debe aprender que "atacar" no es una decision unica, sino que elegir el arma correcta segun el contexto es parte central de la estrategia.



#### (c) Funcion de Recompensa R

La funcion de recompensa incorpora los mismos componentes generales (dano infligido, dano recibido, uso eficiente de recursos, victoria y derrota), pero ajustados a las interacciones especificas del sistema:

$$
R = w_1 \cdot daño\_efectivo\_infligido - w_2 \cdot daño\_recibido + w_3 \cdot uso\_eficiente\_recursos + w_4 \cdot aturdimiento\_logrado + w_5 \cdot victoria - w_6 \cdot derrota
$$

Donde:

- **daño_efectivo_infligido:** depende del arma usada y de la proteccion del enemigo (por ejemplo, Espada contra Hombreras produce un dano_efectivo muy bajo, aunque el "dano nominal" del arma sea alto).
- **daño_recibido:** penalizacion por el dano que el agente recibe, incluyendo el dano evitado si el enemigo queda aturdido.
- **uso_eficiente_recursos:** recompensa por usar la unica Hierba Roja en el momento adecuado (por ejemplo, cuando la salud propia es critica) en lugar de desperdiciarla.
- **aturdimiento_logrado:** recompensa adicional por aturdir al enemigo con el Garrote, ya que abre una ventana de turno gratuito.
- **victoria / derrota:** recompensa o penalizacion grande al finalizar el combate.

**Ejemplo numerico:**

- +2 por atacar con Espada a un enemigo con Hombreras (dano casi anulado).
- +8 por atacar con Garrote (dano moderado, ignora proteccion).
- +6 por usar Bola de Fuego (ignora proteccion, sin aturdir).
- +6 adicional si el Garrote logra aturdir al enemigo.
- -15 por recibir Bola de Fuego del enemigo sin haberlo interrumpido.
- +10 por usar la Hierba Roja en un momento critico; +0 o penalizacion leve si se usa de forma desperdiciada.
- +100 por ganar el combate, -100 por perderlo.

**Justificacion:**
Esta funcion de recompensa obliga al agente a aprender la interaccion entre armas y proteccion (no toda "arma fuerte" es buena en todo momento), a valorar el aturdimiento como una ventaja tactica y no solo como dano directo, y a administrar cuidadosamente su unica Hierba Roja frente a las dos del enemigo.



#### (d) Tipo de Entorno

El entorno es **parcialmente observable**.

$$
O_t \neq S_t
$$

**Justificacion:**

- El agente puede observar que proteccion lleva puesta el enemigo en el turno actual (por ejemplo, Hombreras), pero no necesariamente sabe con certeza cuando el enemigo decidira usar Bola de Fuego, cambiar de estrategia, o si su Garrote esta listo para aturdir.
- El agente no conoce con certeza cuantas Hierbas Rojas exactas piensa usar el enemigo en un turno dado, ni el momento en que decidira curarse.
- Puede haber informacion oculta relacionada con cooldowns de habilidades (por ejemplo, si la Bola de Fuego del enemigo esta disponible o en enfriamiento) que el agente debe inferir a partir del historial de acciones, no observar directamente.

Por lo tanto, el agente toma decisiones con informacion incompleta sobre el estado interno del oponente, lo que incrementa la complejidad del problema y hace mas relevante planificar con base en probabilidades y patrones observados, en lugar de certezas absolutas.

---


### Pregunta 2

Para el sistema que eligió, argumente qué valor de 𝛾sería más apropiado y por qué. Incluya al menos
un ejemplo numérico que ilustre las consecuencias de elegir 𝛾alto versus 𝛾bajo para ese dominio
específico.

#### Respuesta 2

#### (e) Factor de Descuento γ — Versión elaborada con el sistema de combate específico

**Problematica**

En el turno $t$, el agente ve que el enemigo **tiene hombreras puestas**. Tiene tres opciones:

* **Acción A — Atacar con Espada:** intuitivamente "la espada corta más fuerte", pero contra hombreras el daño real es mínimo.
* **Acción B — Atacar con Garrote:** hace menos daño bruto, pero ignora la armadura **y** aturde al enemigo, evitando que use Bola de Fuego el próximo turno.
* **Acción C — Usar Bola de Fuego:** ignora armadura, buen daño, pero no aturde (el enemigo puede contraatacar con su propio Garrote o Bola de Fuego).

Definimos recompensas aproximadas por turno:

$$
R(\text{Espada vs hombreras}) = +2 \quad (\text{daño casi anulado por la armadura})
$$
$$
R(\text{Garrote}) = +5 \ (\text{daño}) \ + \ \text{aturdimiento (evita -15 del contraataque enemigo)}
$$
$$
R(\text{Bola de Fuego enemiga si no es interrumpida}) = -15 \ (\text{daño recibido})
$$

---

**Caso 1: γ → 0.1 (agente miope)**

El agente solo mira el número de daño "nominal" de cada arma, sin proyectar el efecto del aturdimiento ni el desgaste de recursos.

$$
G_t(A) = 2 + 0.1(-15) = 2 - 1.5 = 0.5
$$
$$
G_t(B) = 5 + 0.1(0) = 5
$$

Aquí, por casualidad, el garrote gana en ambos horizontes — pero el problema real aparece en la **planificación a varios turnos**, no en este único paso. Extendamos a 2 turnos:

**Secuencia A (insistir con Espada varios turnos, ignorando que no sirve contra hombreras):**

$$
G_t(A) = 2 + \gamma(-15) + \gamma^2(2) + \gamma^3(-15)+\dots
$$

Con $\gamma = 0.1$:
$$
G_t(A) \approx 2 + 0.1(-15) + 0.01(2) + 0.001(-15) \approx 0.53
$$

El agente miope **no penaliza lo suficiente** el hecho de que va a seguir recibiendo Bola de Fuego turno tras turno mientras insiste con un arma inefectiva. Como el descuento castiga tan fuerte el futuro, **casi no le importa** que esta mala racha se repita — cada golpe de -15 futuro casi no cuenta. El agente puede quedar "atascado" repitiendo la Espada porque el error de cada turno individual parece pequeño.

---

**Caso 2: γ → 0.9 (visión estratégica, varios turnos)**

**Secuencia B (Garrote para aturdir → Bola de Fuego mientras el enemigo no puede reaccionar → conservar la única Hierba Roja para el momento crítico):**

$$
G_t(B) = \underbrace{5}_{\text{turno } t:\ \text{Garrote, aturde}} + \gamma\underbrace{(12)}_{t+1:\ \text{Bola de Fuego libre, enemigo aturdido}} + \gamma^2\underbrace{(-8)}_{t+2:\ \text{enemigo se recupera y contraataca}}
$$

Con $\gamma = 0.9$:
$$
G_t(B) = 5 + 0.9(12) + 0.81(-8) = 5 + 10.8 - 6.48 = 9.32
$$

**Secuencia A (seguir con Espada, ineficaz contra hombreras):**

$$
G_t(A) = 2 + 0.9(-15) + 0.81(2) = 2 - 13.5 + 1.62 = -9.88
$$

Con $\gamma = 0.9$, la diferencia es enorme: $G_t(B) = 9.32$ vs $G_t(A) = -9.88$. El agente **sí** anticipa que:

1. Insistir con la Espada contra un enemigo con hombreras es una trampa (bajo daño repetido + recibir Bola de Fuego sin interrupción).
2. El combo **Garrote (aturde) → Bola de Fuego (daño real, sin armadura de por medio)** es netamente superior, porque el aturdimiento "compra" un turno gratis de daño sin represalia.
3. Como solo tiene **1 Hierba Roja** contra las **2** del enemigo, el agente de horizonte largo también entiende que **prolongar el combate lo perjudica más a él que al enemigo** (el enemigo puede sobrevivir errores gracias a su doble curación). Por eso prefiere secuencias que **terminen el combate rápido** (aturdir + golpe fuerte) en vez de intercambios lentos e inciertos donde el desgaste de recursos lo termina condenando.


**¿Por qué esto no usar γ bajo?**

Con γ bajo, el agente evalúa cada arma casi como si fuera un problema de "daño por turno" aislado, y **nunca conecta**:

* que el aturdimiento del Garrote genera una ventana de daño futuro más grande,
* que la Espada es un arma "trampa" cuyo costo de oportunidad solo se paga en turnos posteriores,
* que la escasez relativa de curación (1 vs 2 hierbas) es una desventaja que se acumula turno a turno y solo importa si el agente "mira lejos".




---
### Pregunta 3

Describa en lenguaje natural una política determinista y una política estocástica para su sistema
elegido. ¿En qué escenario preferiría una política estocástica sobre una determinista?


### Respuesta 3

#### (f) Definir politica


**Política determinista para el sistema de combate**

Una política determinista es una regla fija: dado un estado exacto, siempre produce la misma acción, sin variación.

Para este sistema, una política determinista podría describirse así en lenguaje natural:

*"Si el enemigo tiene Hombreras equipadas, atacar siempre con Garrote. Si el enemigo no tiene Hombreras, atacar siempre con Espada. Si la salud propia cae por debajo del 30%, usar siempre la Hierba Roja. En cualquier otro caso, usar Bola de Fuego."*

Es decir, la política es una función que asigna exactamente una acción a cada estado:

$$
\pi(s) = a
$$

Cada vez que el agente se encuentra en el mismo estado (mismo enemigo, misma salud, misma protección equipada), toma exactamente la misma decisión. No hay margen de variación ni de sorpresa: el comportamiento del agente es completamente predecible una vez que se conoce su estado.

**Política estocástica para el sistema de combate**

Una política estocástica no asigna una única acción a cada estado, sino una distribución de probabilidad sobre las acciones posibles. El agente "elige al azar" entre varias opciones razonables, con ciertas probabilidades asignadas a cada una.

Ejemplo: 

*"Si el enemigo tiene Hombreras equipadas, atacar con Garrote el 70% de las veces y usar Bola de Fuego el 30% restante. Si la salud propia está por debajo del 30%, usar la Hierba Roja el 60% de las veces, pero también existe un 40% de probabilidad de arriesgarse con un ataque directo si se estima que se puede ganar el intercambio."*

Formalmente:

$$
\pi(a \mid s) = P(A_t = a \mid S_t = s)
$$

Aquí, el agente no repite siempre la misma jugada frente al mismo estado; introduce variabilidad deliberada en su comportamiento.

#### En qué escenario preferiría una política estocástica sobre una determinista

Preferiría una política estocástica principalmente en un escenario de **combate contra un oponente que aprende o se adapta al patrón del agente** (por ejemplo, otro jugador humano, o una IA enemiga que observa el historial de acciones).

La razón es la siguiente: si el agente sigue una política determinista del tipo "siempre que el enemigo tenga Hombreras, ataco con Garrote", el enemigo puede detectar este patrón después de unos pocos turnos y anticiparse, por ejemplo reservando su propio Garrote para aturdir primero al agente antes de que este pueda actuar, o cambiando de protección en el momento exacto en que sabe que vendrá el Garrote. La política determinista se vuelve explotable: es predecible y, por lo tanto, vulnerable a la contra-estrategia del rival.

Con una política estocástica, el agente introduce incertidumbre en su propio comportamiento. Aunque el enemigo observe muchos combates, no puede anticipar con certeza qué acción tomará el agente en un estado dado, solo puede estimar probabilidades. Esto es especialmente importante en un juego de combate donde el "adversario" (ya sea otro jugador o una IA rival) puede observar y explotar patrones repetitivos, un problema clásico en teoría de juegos conocido como necesidad de una estrategia de equilibrio mixto: si la política fuera perfectamente predecible, el rival siempre tendría una contra-jugada óptima.

En cambio, en un escenario de un solo jugador contra un entorno fijo que no aprende ni se adapta (por ejemplo, un enemigo controlado por reglas simples sin memoria de acciones pasadas), una política determinista suele ser preferible, ya que una vez que el agente encontró la mejor acción para cada estado, no hay ninguna ventaja en variar su comportamiento; solo introduciría subóptimo adicional sin beneficio estratégico.

